In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("knkarthick/dialogsum")

In [ ]:
ds

In [ ]:
ds['train'][1]

In [ ]:
ds['train'][4]

In [ ]:
ds['train'][1]['dialogue']

In [ ]:
ds['train'][1]['summary']

#**1.Without Fine Tune**

In [ ]:
pip install transformers==4.40.0

In [ ]:
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

In [ ]:
article = ds['train'][1]['dialogue']

In [ ]:
article

In [ ]:
pipe(article,max_length=25,min_length=15,do_sample=False)

In [ ]:
pipe(article,max_length=45,min_length=15,do_sample=False)

#**2.With Fine Tuning**

In [ ]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
# Tokenization
def preprocess_function(batch):
  source = batch['dialogue']
  target = batch['summary']
  source_id = tokenizer(source,padding="max_length",truncation=True,max_length=128)
  target_id = tokenizer(target,padding="max_length",truncation=True,max_length=128)

  labels=target_id['input_ids']
  labels= [[(label if label != tokenizer.pad_token_id else -100) for label in labels_example] for labels_example in labels]

  return{
      "input_ids":source_id["input_ids"],
      "attention_mask":source_id["attention_mask"],
      "labels":labels
  }

In [ ]:
df_source = ds.map(preprocess_function,batched=True)

In [ ]:
pip uninstall transformers peft accelerate -y

In [ ]:
pip install transformers==4.40.0 peft==0.10.0 accelerate==0.30.0

In [ ]:
from transformers import TrainingArguments, Trainer

training_arguments = TrainingArguments(
    output_dir="/content",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    remove_unused_columns = True,
    report_to="none"   # ✅ disables wandb(weight and bias)
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_arguments,
    train_dataset = df_source['train'],
    eval_dataset = df_source['test']
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()

In [ ]:
eval_results

In [ ]:
model.save_pretrained('/content/model_directory')
tokenizer.save_pretrained('/content/model_directory')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('/content/model_directory')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/model_directory')

def summarize(blog_post):
  # tokenize the input blog post
  inputs = tokenizer(blog_post, max_length=1024, truncation=True, return_tensors='pt')

  # generate the summary
  summary_ids = model.generate(inputs["input_ids"],max_length=150,min_length=40,length_penalty=2.0,num_beams=4,early_stopping=True)

  # decode the summary
  summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

  # clean output
  summary = summary.replace("#", "").strip()

  return summary

In [ ]:
blog_post="""
Machine learning is a transformative branch of artificial intelligence that enables computers to learn from data and improve
their performance without being explicitly programmed. It plays a crucial role in modern technology, powering applications
across industries such as healthcare, finance, education, and entertainment. By analyzing vast amounts of structured and
unstructured data, machine learning algorithms can detect patterns, make predictions, and automate complex decision-making
processes. For example, in healthcare, machine learning models assist doctors in diagnosing diseases at early stages,
while in finance, they are used for fraud detection and risk assessment. Recommendation systems used by streaming platforms
and e-commerce websites also rely heavily on machine learning to provide personalized user experiences.

There are several types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning.
Supervised learning involves training a model on labeled data, where the correct output is known, while unsupervised learning
deals with unlabeled data and aims to find hidden patterns or groupings. Reinforcement learning, on the other hand, focuses on
training agents to make decisions by rewarding desired behaviors. With the advancement of deep learning, a subset of machine
learning that uses neural networks with multiple layers, the field has seen significant breakthroughs in areas like computer
vision, speech recognition, and natural language processing.

Despite its many advantages, machine learning also presents challenges. Issues such as data privacy, algorithmic bias, and
lack of transparency can impact the reliability and fairness of models. Additionally, training complex models often requires
large computational resources and high-quality data. As a result, researchers and practitioners are actively working on
developing more efficient, interpretable, and ethical machine learning systems. Overall, machine learning continues to evolve
rapidly, shaping the future of technology and offering innovative solutions to real-world problems.
"""
summary=summarize(blog_post)

print(f"Summary : {summary}")

In [ ]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load model
model_path = "/content/model_directory"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Summarization function
def summarize(text):
    if not text.strip():
        return "Please enter some text."

    text = "summarize: " + text

    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # clean output
    summary = summary.replace("#", "").strip()

    return summary


# UI Layout
with gr.Blocks() as app:

    gr.Markdown("# 📝 AI Text Summarization App")
    gr.Markdown("Enter a long paragraph and get a short summary")

    # Input box (full width)
    input_text = gr.Textbox(
        lines=12,
        placeholder="Enter your text here...",
        label="Input Text"
    )

    # Buttons row
    with gr.Row():
        submit_btn = gr.Button("🚀 Summarize")
        clear_btn = gr.Button("🧹 Clear")

    # Output box (full width)
    output_text = gr.Textbox(
        lines=10,
        label="Summary"
    )

    # Button actions
    submit_btn.click(fn=summarize, inputs=input_text, outputs=output_text)

    clear_btn.click(lambda: ("", ""), None, [input_text, output_text])


# Launch app
app.launch()

In [ ]:
import shutil
shutil.make_archive("Model","zip","/content/model_directory")

In [ ]:
from google.colab import files
files.download("Model.zip")